In [1]:
!git clone https://github.com/OpenMOSS/MOSS-TTS.git
%cd MOSS-TTS
!pip install datasets soundfile matplotlib numpy -q wandb
!apt-get install -y ffmpeg -q

fatal: destination path 'MOSS-TTS' already exists and is not an empty directory.
/content/MOSS-TTS
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [2]:
# MODEL_NAME  = "OpenMOSS-Team/MOSS-TTS" # Delay 8B
MODEL_NAME = "OpenMOSS-Team/MOSS-TTS-Local-Transformer" # Local 1.7B
MAX_NEW_TOKENS = 200 
texts = ["The weather is so nice today and the birds are singing in the trees.", "The weather is so nice today and the birds are singing in the trees. "]

In [3]:
import importlib.util
import wandb

import json
import random
import time
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from transformers import AutoModel, AutoProcessor
import time
import torch
import numpy as np
import json
from pathlib import Path

# required SDPA backend flags from official MOSS-TTS docs
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
#load model and processor
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code= True)

if hasattr(processor, 'audio_tokenizer'):
    processor.audio_tokenizer = processor.audio_tokenizer.to(device).eval()

model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code= True, torch_dtype= torch.bfloat16).to(device).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/1600 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/556 [00:00<?, ?it/s]

In [7]:
#use pytorch profiler to get detailed breakdown of time taken by each component of the model
for i, text in enumerate(texts):

    #warm up on first sample
    if i == 0:
        inputs = processor([{"role": "user", "content": text}], return_tensors="pt").to(device)

        with torch.inference_mode():
            model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    else:
        inputs = processor([{"role": "user", "content": text}], return_tensors="pt").to(device)

        with torch.inference_mode():

            with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]) as profile:
                model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
        

profile.export_chrome_trace(f"hf_profile_{'delay' if MODEL_NAME == 'OpenMOSS-Team/MOSS-TTS' else 'local'}.json")

print(profile.key_averages().table(sort_by="cuda_time_total"))


10it [00:02,  3.36it/s]
10it [00:04,  2.46it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::matmul         2.39%      69.645ms        23.54%     685.887ms      50.619us       0.000us         0.00%     255.423ms      18.850us         13550  
                                           aten::linear         1.25%      36.358ms        27.88%     812.266ms      59.990us       0.000us         0.00%     255.392ms      18.862us         13540  
         